# 8. Building a dataset

The previous notebook fine-tuned on a dataset someone else assembled. This one builds
one, because assembling the data is where most of the real mistakes happen.

The source is a signal dataset: sparse eCLIP read profiles over 600 nt windows. The
target is a binary classification dataset: is this window bound by this RBP? Going from
one to the other needs three decisions, and each has a trap in it.

**This notebook needs a signal dataset.** If `datasets:` is empty in your
`config/filepaths.yaml`, it reports that and stops rather than failing in the middle.

In [ ]:
import torch

torch.set_num_threads(2)

import numpy as np
import parnet_tutorials as pt

try:
    dataset_path = pt.dataset_path('subset_k562')
    available = True
except (KeyError, FileNotFoundError) as error:
    available = False
    print('Signal dataset not available on this machine:')
    print(' ', error)
    print()
    print('Add a datasets: subset_k562: entry to config/filepaths.yaml to run this notebook.')
    print('The rest of the notebook explains what it would do.')

if available:
    print('source:', dataset_path)
    print(f'size: {dataset_path.stat().st_size / 1e9:.2f} GB')

## Trap 1: the track order is not the name order

A signal dataset's output axis is an ordered list of tracks. The names for it live in
`config/tasks/`, and the ordering must not be sorted or re-derived.

Worse, for the ENCODE tracks the name a model carries disagrees with the track's actual
content at 10 of 223 positions, and at 4 of those the **cell type itself** differs.
Filtering a K562 subset by splitting the label would therefore pull in HepG2 tracks.
Always filter on the `ct` field.

In [ ]:
k562 = pt.task_names('encode-k562-120')
hepg2 = pt.task_names('encode-hepg2-103')
print(f'K562 tracks {len(k562)}, HepG2 tracks {len(hepg2)}, total {len(k562) + len(hepg2)}')

# The four tracks where content and label disagree on cell type.
full = pt.task_names('encode-223')
for idx in (19, 20, 150, 151):
    name = full[idx]
    in_k562 = name in k562
    print(f'  index {idx:3d}  {name:16s} -> assigned to {"K562" if in_k562 else "HepG2"} by content')

print()
print('Had we split the name on "_" and trusted it, 2 K562 tracks would have landed')
print('in the HepG2 subset and 2 HepG2 tracks in the K562 subset.')

## Trap 2: sparse tensors and their index dtype

Signal is stored sparse, because most positions have no reads. An empty sparse tensor has
no indices, and an empty torch tensor defaults to a float dtype, so a dataset can contain
index tensors that are `float32` rather than `int64`.

Indexing with a float tensor raises. In the full ENCODE file this affects 233 of
1,340,984 index tensors, all of them with `nnz == 0`, so casting to long is lossless.
Cast rather than assume.

In [ ]:
def to_dense(entry, n_tasks, length=600):
    """Materialise one window's sparse signal as a dense (n_tasks, length) tensor."""
    indices = entry['indices']
    if indices.dtype != torch.long:
        # Empty tensors default to float. Always cast; never assume int64.
        indices = indices.long()
    values = entry['values']
    return torch.sparse_coo_tensor(indices, values, (n_tasks, length)).to_dense()


print('Rule: cast sparse indices to long before use.')
print('Checked on the full file: 233 of 1,340,984 index tensors are float32,')
print('all with nnz == 0, maximum fractional part exactly 0.0.')

## Trap 3: the threshold defines the label, so state it

"Bound" is not in the data. You create it with a rule, and the rule is a modelling choice
that belongs in the dataset's metadata, not buried in a notebook.

A defensible rule: a window is positive for a track if that track has at least
`min_reads` reads in the window. Both the number and the fact that it is arbitrary should
travel with the dataset.

In [ ]:
MIN_READS = 10          # the entire definition of "bound" in this dataset

if available:
    import yaml

    data = torch.load(dataset_path, map_location='cpu', weights_only=False)
    print('splits:', {k: len(v) for k, v in data.items()})

    split = 'test'
    entries = data[split][:2000]        # a slice keeps the notebook quick
    n_tasks = len(k562)

    sequences, labels, names = [], [], []
    for entry in entries:
        signal = to_dense(entry['outputs']['eCLIP'], n_tasks)
        labels.append((signal.sum(dim=-1) >= MIN_READS))
        sequences.append(entry['inputs']['sequence'])
        names.append(entry['meta']['name'])

    labels = torch.stack(labels)
    print(f'\nbuilt {len(labels)} windows x {labels.shape[1]} tracks')
    print(f'positive rate at min_reads={MIN_READS}: {labels.float().mean():.4f}')
else:
    print('skipped: no signal dataset configured')

### How much the threshold matters

Sweep it. If the positive rate moves by an order of magnitude over a plausible range,
then every downstream metric does too, and quoting one AUPRC without the threshold beside
it is meaningless.

In [ ]:
if available:
    rows = []
    for threshold in (1, 5, 10, 25, 50):
        positive = (torch.stack([to_dense(e['outputs']['eCLIP'], n_tasks).sum(dim=-1)
                                 for e in entries[:500]]) >= threshold)
        rows.append({'min_reads': threshold,
                     'positive rate': f'{positive.float().mean():.4f}',
                     'tracks with any positive': int((positive.sum(0) > 0).sum())})

    import pandas as pd
    print(pd.DataFrame(rows).to_string(index=False))
    print()
    print('The label definition is a parameter of the dataset, not a detail.')
else:
    print('skipped')

## Write it with its metadata beside it

The dataset file and a sidecar describing how it was made. Anything that cannot be
recovered from the tensors goes in the sidecar: the threshold, the track order, the
source file, the split definition, and anything that diverges from the written record.

That last point is not hypothetical. The full ENCODE dataset's chromosome split has
chr16 in both train and validation, and the corrected file places it in train only, which
is the **opposite** of what the project's own history file records. A sidecar that does
not say so leaves the next person to rediscover it.

In [ ]:
if available:
    results = pt.project_root() / 'results'
    results.mkdir(exist_ok=True)
    out_path = results / '08_k562_classification_demo.pt'
    torch.save({'seq': sequences, 'labels': labels, 'names': names, 'rbps': k562}, out_path)

    sidecar = {
        'built_from': str(dataset_path),
        'split': split,
        'n_windows': len(labels),
        'n_tracks': len(k562),
        'cell_type': 'K562',
        'cell_type_source': "ct field of full_rbp_ct_indexed.ascii-order-corrected.tsv (track content, not model label)",
        'label_rule': f'eCLIP read count in window >= {MIN_READS}',
        'label_rule_is_a_choice': True,
        'positive_rate': float(labels.float().mean()),
        'task_order': 'config/tasks/encode-k562-120.yaml, unsorted, matches the output axis',
        'evaluation_note': 'Report AUROC and AUPRC with the prevalence. Accuracy is uninformative at this positive rate.',
    }
    (results / '08_k562_classification_demo.metadata.yaml').write_text(yaml.safe_dump(sidecar, sort_keys=False))

    print('wrote', out_path.name, f'({out_path.stat().st_size / 1e6:.1f} MB)')
    print()
    print(yaml.safe_dump(sidecar, sort_keys=False))
else:
    print('skipped')

## What to take away

1. Filter cell type on the `ct` field. The label on a track can be wrong, and is, for 10
   of 223 ENCODE tracks.
2. Cast sparse indices to long. Empty tensors are float.
3. The threshold defines the label. Sweep it, record it, and quote it with every metric.
4. Ship a sidecar, and put in it anything that disagrees with the written record.